<a href="https://colab.research.google.com/github/Song-yiJung/korean-ocr-lectures/blob/main/2026-08-aks-lecture/02_step2_gemini/step2_gemini_colab_20260813.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 2 — Gemini로 문맥 교정하기**

*2026 한국학중앙연구원 OCR 강의 실습*

step1이 뽑은 1차 텍스트를 문맥에 맞게 고치는 단계다.

step1의 Vision은 글자의 **모양**만 보았다. 그래서 행 순서가 흐트러지고, 닮은 글자를 잘못 읽고, 표의 칸 구조가 뭉개졌다. Gemini는 그 결과를 **원본 이미지와 함께** 다시 보고 고친다. 글자만 보는 것이 아니라 문장으로 읽기 때문에 "이 자리에 올 말은 이것"이라는 판단을 한다.

**직접 고치는 셀은 ③과 ⑤ 둘이다.** 강의에서는 ③의 키 이름만 확인하고 ⑤는 그대로 둔다.

| 셀 | 하는 일 | 손대는가 | 시간 |
|---|---|---|---|
| ① | 프로그램 설치 | 실행만 | 30초 |
| ② | Drive 연결 | 실행만 | 20초 |
| **③** | **설정** | **★ 고친다** | — |
| ④ | 인증 | 실행만 | 5초 · **확인 1** |
| **⑤** | **교정 지침** | **★ 읽는다** | — |
| ⑥ | 대상 점검 | 실행만 | 즉시 · **확인 2** |
| ⑦ | 교정 실행 | 실행만 | 약 2분 |
| ⑧ | 1차·2차 대조 | 실행만 | 즉시 · **확인 3** |
| ⑨ | 비용 확인 | 실행만 | 즉시 |

**⑤가 이 노트북의 알맹이다.** 코드가 아니라 한국어로 쓴 지침이고, 결과의 질을 가장 크게 좌우한다.

## 시작 전

### 1. 사본 저장
상단 **`파일 → Drive에 사본 저장`**을 누른다. 이후 작업은 사본에서 한다.

### 2. step1을 먼저 끝냈어야 한다
`OUTPUT_DIR` 폴더에 사료별 `.json`이 만들어져 있어야 한다. 없으면 이 노트북은 할 일이 없다.

### 3. Gemini API 키

[API 키 발급 안내](https://github.com/Song-yiJung/korean-ocr-lectures/blob/main/docs/api-key-setup.md)의 절차를 **강의 전날까지** 마쳐 둔다. 그 절차대로 하면 결제 계정이 연결된 프로젝트 안에서 키가 만들어지므로, 별도로 할 일은 없다.

**이 실습의 Gemini 키는 무료 등급이다.** 결제 계정을 연결하지 않은 별도 프로젝트에서 만든 키이므로 요금이 청구되지 않는다. 호출 한도는 프로젝트 단위로 적용되므로, 여러 사람이 각자 자기 키로 동시에 실행해도 서로 막지 않는다.

다만 **무료 등급에서는 입력한 내용이 구글의 모델 개선에 사용된다.** 이 실습의 사료 3장은 이미 공개된 공공데이터라 문제가 되지 않지만, **본인의 미간행 소장 자료를 다룰 때는 유료 등급으로 바꾸어 쓴다.** 소장기관과의 이용 조건에 걸릴 수 있기 때문이다. 자세한 것은 발급 안내 6절에 있다.

step1의 Vision 키는 결제 계정이 연결된 프로젝트에 있고, step2의 Gemini 키는 결제를 연결하지 않은 프로젝트에 있다. **두 키는 서로 다른 프로젝트에 있다.**

무료 등급이라 이 실습에서는 요금이 들지 않는다. 유료 등급이었다면 얼마였을지는 ⑨에서 확인한다.

> **step1 키와 생김새가 다르다.** step1의 Vision 키는 컴퓨터에 내려받아 Drive에 올려 둔 **파일**이지만, step2의 Gemini 키는 **문자열 한 줄**이다. 그래서 보안 비밀에 넣는 것도 다르다. step1은 *파일이 있는 경로*를, step2는 *키 문자열 자체*를 넣는다. **이 둘을 바꿔 넣는 것이 가장 흔한 실수다.**

### 4. ★ 폴더 경로를 step1과 똑같이 ★
③의 `INPUT_DIR`·`OUTPUT_DIR`은 **step1에서 쓴 값과 글자 그대로 같아야 한다.** 다르면 step2가 원본 이미지나 step1 결과를 찾지 못한다. step1 노트북을 열어 두고 옮겨 적는 편이 안전하다.

3장 기준 약 1분.

## 코드를 읽는 법

step1에서 본 것(`import`, `이름 = 값`, `def`, `for`, `if`)이 그대로 다시 나온다. **여기서 새로 보이는 것은 셋뿐이다.**

| 보이는 것 | 뜻 |
|---|---|
| `data['vision_raw']` | **사전에서 값을 꺼낸다.** step1이 만든 JSON에서 `vision_raw` 칸의 내용을 가져온다는 뜻이다. |
| `A + B` | 문자열을 **이어 붙인다.** ⑦에서 교정 지침과 1차 텍스트를 하나로 붙여 Gemini에 보낸다. |
| `time.sleep(SLEEP_SEC)` | **그만큼 쉰다.** 요청을 한꺼번에 몰아 보내지 않으려고 간격을 둔다. ③에서 6초로 정해 두었다. |

`client`라는 이름도 다시 나온다. step1에서 Vision과 이야기하는 창구를 `client`로 만들었던 것처럼, 여기서는 Gemini와 이야기하는 창구를 같은 이름으로 만든다. **두 노트북의 구조가 같다.** 창구를 하나 만들고, 그 창구에 대고 한 장씩 요청한다.

크게 다른 점이 하나 있다. **보내는 것이 글자만이 아니다.**

```python
client.models.generate_content(model=..., contents=[prompt, image])
```

`contents`의 대괄호 안에 **지시문과 이미지가 함께** 들어간다. Gemini는 둘을 한꺼번에 받아, 사진을 보면서 지시를 따른다. 이렇게 글과 이미지를 같이 다루는 것을 **멀티모달**이라고 부른다. step1의 Vision이 이미지만 받았던 것과 다르다.

### ① [실행만] 프로그램 설치

Gemini를 부르는 데 필요한 프로그램을 설치한다.

`google-genai`는 구글이 배포하는 파이썬 묶음으로, step1의 `google-cloud-vision`과 짝을 이루는 것이다. `pillow`는 이미지를 열어 다루는 도구다. Gemini에 사진을 함께 보내야 해서 필요하다.

> 예전에는 `google-generativeai`라는 다른 묶음을 썼다. 구글이 2025년에 통합 묶음으로 바꾸면서 옛것은 지원이 끝났다. 인터넷에서 찾은 코드가 `genai.configure(...)`로 시작한다면 옛 방식이니 그대로 쓰지 않는다.

In [ ]:
!pip install -q google-genai pillow

### ② [실행만] Google Drive 연결

step1에서 만든 결과와 원본 이미지가 Drive에 있으므로 다시 연결한다.

```
Mounted at /content/drive
```

이미 연결돼 있으면 그 줄만 나오고 권한 창은 뜨지 않는다.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### ③ ★ 여기를 고친다 — 설정

**강의에서는 [1]만 확인하고 나머지는 그대로 두고 실행한다.**

---

#### [1] `GEMINI_KEY_SECRET_NAME` — 키를 담아 둘 보안 비밀의 이름

step1과 같은 방식이다. 화면 **맨 왼쪽 🔑 아이콘 → ＋ 새 보안 비밀**을 누른다.

1. **이름**에 `GEMINI_API_KEY` 라고 적는다. *(아래 코드의 값과 글자 하나까지 같아야 한다.)*
2. **값**에 발급받은 키 문자열을 붙여 넣는다. 준비 단계에서 `gemini_key.txt`에 저장해 둔 그 한 줄이다.
3. **노트북 액세스**를 **켠다.**

> step1에서는 값에 *파일 경로*를 넣었지만 여기서는 *키 문자열 자체*를 넣는다. step1의 보안 비밀은 그대로 두고, **새로 하나 더 만드는 것**이다. 왼쪽 🔑 목록에 두 개가 나란히 보이면 맞다.

---

#### [2] `INPUT_DIR`·`OUTPUT_DIR` — step1과 똑같아야 한다

`INPUT_DIR`은 원본 이미지 폴더다. Gemini가 사진을 다시 보아야 하므로 필요하다. `OUTPUT_DIR`은 step1이 만든 JSON이 있는 폴더이고, 교정 결과도 같은 파일에 채워진다.

**step1에서 값을 바꿨다면 여기도 똑같이 바꾼다.** 이것이 step2에서 가장 자주 나는 오류다.

#### [3] `MODEL_NAME` — 쓸 모델

`gemini-2.5-flash`로 충분하다. 흘림체 서신이 많은 자료군이면 `gemini-2.5-pro`가 낫지만 느리고 비싸다.

#### [4] `SLEEP_SEC` — 호출 간격

호출과 호출 사이에 두는 간격이다. **무료 등급은 분당 호출 수 한도가 유료 등급보다 낮으므로 6초로 둔다.** 3장이면 총 18초가 더 걸릴 뿐이다. 자세한 것은 ⑦에서 설명한다.

In [ ]:
# ③ 설정 — 강의에서는 [1]만 확인하고 실행한다

# [1] 왼쪽 🔑 에 등록한 보안 비밀의 이름 (값에는 키 문자열 자체를 넣는다)
GEMINI_KEY_SECRET_NAME = 'GEMINI_API_KEY'

# [2] step1에서 쓴 것과 글자 그대로 같아야 한다
INPUT_DIR = '/content/drive/MyDrive/사료_이미지_강의'
OUTPUT_DIR = '/content/drive/MyDrive/vision_결과_강의'

# [3] 쓸 모델
MODEL_NAME = 'gemini-2.5-flash'

# [4] 호출 간격(초) — 무료 등급이므로 넉넉히 둔다
SLEEP_SEC = 6

print('이미지 폴더:', INPUT_DIR)
print('결과 폴더  :', OUTPUT_DIR)
print('모델       :', MODEL_NAME)

### ④ [실행만] 인증 — **확인 1**

③에서 등록한 보안 비밀을 읽어 Gemini에 연결하고, 폴더가 제대로 잡혔는지 함께 확인한다.

**아래 네 줄이 나오면 통과다.**

```
Gemini 연결 완료: gemini-2.5-flash
이미지 폴더 확인: 3 장
step1 결과 확인: 3 개
```

`이미지 폴더를 찾지 못했습니다` 또는 `step1 결과 폴더를 찾지 못했습니다`가 나오면 ③의 경로가 step1과 다른 것이다. **여기서 잡아야 한다.** 그냥 넘어가면 ⑦에서 아무것도 처리하지 못한 채 조용히 끝난다.

`SecretNotFoundError`라는 붉은 오류가 나면 보안 비밀 이름이 다르거나 노트북 액세스가 꺼진 것이다.

`genai.Client(api_key=...)`가 Gemini와 이야기할 창구를 만드는 줄이다. step1에서 `vision.ImageAnnotatorClient()`로 만들었던 것과 같은 자리이며, 이 `client`를 ⑦에서 다시 쓴다.

In [ ]:
import os
from google import genai
from google.colab import userdata

# 보안 비밀에서 키 문자열을 꺼낸다
api_key = userdata.get(GEMINI_KEY_SECRET_NAME)

# Gemini와 이야기할 창구를 만든다
client = genai.Client(api_key=api_key)
print('Gemini 연결 완료:', MODEL_NAME)

# 폴더가 step1과 맞는지 확인한다
if os.path.isdir(INPUT_DIR):
    n_img = len(os.listdir(INPUT_DIR))
    print('이미지 폴더 확인:', n_img, '장')
else:
    print('이미지 폴더를 찾지 못했습니다:', INPUT_DIR)
    print('  → ③의 INPUT_DIR이 step1과 같은지 확인하세요.')

if os.path.isdir(OUTPUT_DIR):
    n_res = len(os.listdir(OUTPUT_DIR))
    print('step1 결과 확인:', n_res, '개')
else:
    print('step1 결과 폴더를 찾지 못했습니다:', OUTPUT_DIR)
    print('  → step1을 먼저 끝냈는지, ③의 OUTPUT_DIR이 맞는지 확인하세요.')

### ⑤ ★ 교정 지침 (SYSTEM_PROMPT)

**이 노트북에서 결과의 질을 가장 크게 좌우하는 곳이다.** 코드가 아니라 한국어로 쓴 지시문이고, Gemini에게 "이 사료를 어떤 기준으로 고쳐라"라고 알려 준다.

아래 기본값은 1930년대 조선총독부 행정문서와 불국사 사료에 맞춰 두었다. **강의에서는 고치지 말고 그대로 실행한다.**

```
교정 지침 길이: 2,970 자
```

---

#### 지금 볼 곳은 7번이다

지침이 열 항목이라 다 읽을 시간은 없다. **7번만 본다.**

> **7. 환각(Hallucination) 방지 — 엄수**
> - 원본에 없는 내용을 생성하지 말 것.
> - 한 글자라도 이미지에서 확신할 수 없으면 해당 글자 뒤에 `[?]` 표기.
> - 완전히 판독 불가능한 구간은 `□` 또는 `(판독불가)`로 표기.

모델은 빈칸을 그냥 두는 것보다 그럴듯하게 채우는 쪽으로 기운다. 그래서 **"모르면 모른다고 하라"고 명시적으로 시켜야 한다.** 이 세 줄이 없으면 판독 불가능한 자리에 자연스러운 글자가 들어가 버리고, 그건 원본과 대조하기 전에는 드러나지 않는다.

⑧에서 결과를 볼 때 `[?]`와 `□`가 실제로 찍혔는지 확인한다. 하나도 없다면 오히려 의심할 일이다.

#### 나머지 항목이 하는 일

1번은 표의 칸 구조를 뭉개지 말라는 것, 5번은 구자체를 신자체로 바꾸지 말라는 것(`佛`을 `仏`로 고치면 사료가 아니게 된다), 6번은 이 자료군에 실제로 나오는 인명·지명·사찰명을 미리 알려 주는 것이다.

#### 본인 사료에 쓸 때

조선시대 한문이나 근대 한글 신문에 적용하려면 **이 지침을 다시 써야 한다.** 시대·언어·서식, 자주 나오는 고유명사, 자주 혼동되는 글자를 본인 자료에 맞게 넣는다.

다만 **길게 쓸수록 좋은 것이 아니다.** 지침을 더 정교하게 다듬어 실측 비교한 결과, 오히려 정확도가 떨어진 사례가 있다. 조건을 많이 걸수록 모델의 주의가 흩어진다. 기본값과 비슷한 분량을 유지하는 편이 안전하다.

In [ ]:
SYSTEM_PROMPT = """당신은 1930년대 일제강점기(1910~1945) 조선총독부 행정문서부터 민간 서신, 전보에 이르는 문화유산 기록물 해독에 통달한 역사학자이자 아키비스트이다.
본 사료는 '불국사(佛國寺)' 및 관련 문화재를 대상으로 한 공문서, 서신, 전보 등으로 일본 한자(구자체), 히라가나, 가타카나, 숫자가 혼용되어 있으며, 인쇄체와 수기체(필기)가 섞여 있다.

첨부된 원본 이미지와 1차 Vision API가 물리적으로 추출한 원시 텍스트(vision_raw)를 교차 검증하여, 다음 학술적 지침에 따라 텍스트를 교정 및 복원한다.

[교정 및 복원 지침]

1. 표·목록 레이아웃 보존 (Tabular Layout Preservation) — 최우선 규칙:
   - 동일 열의 값이 반복될 때는 절대 평탄화하지 말고 반드시 '〃' 또는 '全' 기호로 표기한다.
   - 행정 계층(郡/面/里)은 각 행에서 동일 상위가 반복되면 '〃'로 축약하되, 상위 계층을 임의로 생략하거나 압축하지 말 것.
   - 열 간 공백은 이미지에 보이는 열 구분을 반영하여 탭/다수 공백으로 유지한다.

2. 일본어 법령체 접속사·조사 정밀 판독:
   - '又ハ'(또는)과 '及'(그리고)를 절대 혼동하지 말 것.
   - 법령·공문서 상투어는 시대 용례를 반영한다 (예: '保存上必要ト認メラルル事項', 'ニ關スル', 'ニ付').
   - 조동사 활용('タル', 'ベキ', 'セシ', 'ラル')을 임의로 단순화하지 말 것.

3. 표준 조사서 템플릿 인식:
   고적·보물 조사서는 다음 8개 고정 필드를 따른다. 번호를 중복하거나 건너뛰지 말 것.
     一、名稱   二、所在地   三、地域、地番、地目及地積   四、工作物其ノ他ノ物件ノ名稱、員數、品質、形狀、構造、形式及大サ
     五、現狀   六、由來又ハ傳說   七、保存上必要ト認メラルル事項   八、其他參考ト爲ルベキ事項

4. 숫자·도량형 정확성:
   - 자릿수 구분 쉼표/점을 반드시 보존한다 (예: '三,三〇九坪'을 '三九坪'으로 축약 금지).
   - 치수 단위(尺·寸·間·段·坪)는 이미지 판독에 근거; 추측 금지.
   - 연호는 '昭和N年M月D日' 형식으로 풀어쓴다.

5. 한자 변이자(구자체) 보존:
   - 현대 일본 상용한자(신자체)로 바꾸지 말 것:
     臺(O)/台(X), 龜(O)/亀(X), 佛(O)/仏(X), 國(O)/国(X), 關(O)/関(X), 舊(O)/旧(X), 藝(O)/芸(X), 圖(O)/図(X)
   - 유사 자형 오독 주의: 開/圓, 螭/龜, 奉/春, 栗/栢, 蒙/豪.

6. 고유명사·사찰명·미술사 용어 복원:
   - 인명: 長尾欽彌, 藤島亥治郞 등 당대 관련자.
   - 지명: 慶州郡, 內東面 馬洞里, 陽北面 凡谷里 등 조선 행정구역.
   - 불교/미술사 용어: 阿彌陀如來, 毘盧舍那佛, 舍利塔, 石窟庵, 多寶塔, 釋迦塔, 白雲橋, 青雲橋, 七寶橋, 蓮華橋.
   - 사찰명은 실제 존재한 이름만: 佛國寺, 石窟庵, 芬皇寺, 鳳停寺, 開心寺, 淨惠寺, 栢栗寺.

7. 환각(Hallucination) 방지 — 엄수:
   - 원본에 없는 내용을 생성하지 말 것.
   - 한 글자라도 이미지에서 확신할 수 없으면 해당 글자 뒤에 '[?]' 표기.
   - 완전히 판독 불가능한 구간은 '□' 또는 '(판독불가)'로 표기.

8. 인장·직인·부기 처리:
   - 붉은 인장이 찍힌 위치에는 '(鈐印)' 또는 '(직인: 내용)' 형태로 표기.
   - 관인 테두리로 인한 무의미한 영문자(DES, DID 등)는 삭제.

9. 문서 레이아웃 복원:
   - 세로쓰기(우→좌) 원칙. vision_raw의 배열은 참고만 하고 이미지의 논리적 흐름에 따라 재구성.
   - 공문서 양식은 원본 서식 그대로 유지.

10. 출력 형식 통제:
    - 부가적인 설명, 주석, 마크다운 코드 블록 절대 금지 — 교정된 순수 텍스트만 반환.
    - 빈 줄은 원본 문서의 단락 구분을 따라 유지.
"""

print('교정 지침 길이:', len(SYSTEM_PROMPT), '자')

### ⑥ [실행만] 대상 점검 — **확인 2**

step1이 만든 JSON을 살펴, 무엇을 교정할지 세 갈래로 나눈다. **실제 호출 전에 문제를 잡는 자리다.**

```
교정할 것        : 3 개
이미 교정된 것   : 0 개
원본 이미지 없음 : 0 개
```

**`교정할 것`이 3개면 통과다.**

`원본 이미지 없음`이 0이 아니면 ③의 `INPUT_DIR`이 step1과 다른 것이다. **여기서 멈추고 고친다.** 이 점검이 없으면 ⑦이 아무 일도 하지 않은 채 "완료"만 찍고 끝나, 성공한 줄 알고 넘어가게 된다.

`이미 교정된 것`은 이 노트북을 두 번째 실행할 때 나온다. 다시 돌려도 돈이 두 번 나가지 않도록 건너뛴다.

코드는 폴더의 파일을 하나씩 열어(`for`), 세 가지를 차례로 따진다(`if`). 이미 `gemini_corrected` 칸이 차 있는가, 원본 이미지가 실제로 있는가, 둘 다 아니면 교정 대상이다. `_full.json`은 좌표 파일이라 처음부터 뺀다.

In [ ]:
import os
import json

targets = []
already = []
no_image = []

for name in sorted(os.listdir(OUTPUT_DIR)):

    # step1이 만든 텍스트 JSON만 본다 (_full.json 은 좌표 파일)
    if not name.endswith('.json') or name.endswith('_full.json'):
        continue

    data = json.load(open(os.path.join(OUTPUT_DIR, name), encoding='utf-8'))

    if data['gemini_corrected'].strip():
        already.append(name)
        continue

    if not os.path.exists(os.path.join(INPUT_DIR, data['file_path'])):
        no_image.append(name)
        continue

    targets.append(name)

print('교정할 것        :', len(targets), '개')
print('이미 교정된 것   :', len(already), '개')
print('원본 이미지 없음 :', len(no_image), '개')

if no_image:
    print()
    print('원본 이미지를 찾지 못한 파일이 있습니다.')
    print('  → ③의 INPUT_DIR이 step1과 같은지 확인하세요.')
    for name in no_image:
        print('   ', name)

### ⑦ [실행만] 교정 실행

**여기가 실제로 Gemini를 부르는 셀이다.** 한 장에 20초쯤 걸린다. 3장이면 1분 남짓이다.

```
[1/3] E006-005-001-001.json
   -> 388 자
[2/3] E006-005-001-002.json
   -> 41 자
[3/3] E006-005-002-001.json
   -> 1102 자

완료: 3 장 교정
```

**화면이 잠깐 멈춘 것처럼 보이는 구간이 있다.** 고장이 아니다. 사료 한 장을 읽고 판독하는 데 20초쯤 걸리는 것이고, 호출과 호출 사이에 6초씩 쉬기도 한다.

이 간격은 요청을 몰아 보내지 않기 위한 것이다. **무료 등급은 분당 호출 수 한도가 낮아 유료 등급보다 넉넉히 두어야 한다.** 다만 호출 한 건 자체가 이미 20초쯤 걸리므로, 사료 3장 규모에서는 한도에 걸릴 일이 거의 없다.

보내는 것은 두 가지다. **⑤의 교정 지침에 step1의 1차 텍스트를 이어 붙인 글**과 **원본 이미지**다. Gemini는 사진을 보면서 1차 텍스트의 어디가 틀렸는지 판단한다.

결과는 두 곳에 저장된다. step1이 비워 둔 `gemini_corrected` 칸이 채워지고, 사람이 읽기 좋게 `_gemini_final.txt`도 따로 만들어진다.

> 도중에 붉은 오류로 멈추면 잠시 기다렸다가 이 셀을 다시 누른다. **⑥이 이미 끝난 장을 걸러 주므로 처음부터 다시 하지 않는다.** 오류 문구에 `429`가 보이면 짧은 시간에 요청이 몰린 것이니 `SLEEP_SEC`을 10으로 올리고 1분쯤 기다렸다가 다시 누른다. 무료 등급에서는 하루 호출 수 한도도 있으므로, 같은 키로 이미 여러 번 돌렸다면 그쪽에 걸린 것일 수도 있다.

In [ ]:
import os
import json
import time
import PIL.Image

done = 0
total_in = 0
total_out = 0

for i, name in enumerate(targets, 1):
    path = os.path.join(OUTPUT_DIR, name)
    data = json.load(open(path, encoding='utf-8'))

    print('[%d/%d]' % (i, len(targets)), name)

    # 보낼 것 두 가지: 지침 + 1차 텍스트, 그리고 원본 이미지
    prompt = SYSTEM_PROMPT + '\n\n[1차 텍스트]\n' + data['vision_raw']
    image = PIL.Image.open(os.path.join(INPUT_DIR, data['file_path']))

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[prompt, image]
    )
    corrected = response.text.strip()

    # (1) step1 JSON 의 빈 칸을 채운다
    data['gemini_corrected'] = corrected
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    # (2) 사람이 읽을 텍스트 파일
    file_id = name.replace('.json', '')
    txt_path = os.path.join(OUTPUT_DIR, file_id + '_gemini_final.txt')
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(corrected)

    total_in = total_in + response.usage_metadata.prompt_token_count
    total_out = total_out + response.usage_metadata.candidates_token_count

    print('   ->', len(corrected), '자')
    done = done + 1

    time.sleep(SLEEP_SEC)

print('\n완료:', done, '장 교정')

### ⑧ [실행만] 1차·2차 대조 — **확인 3**

**이 실습에서 가장 중요한 화면이다.** 같은 사료의 1차(Vision)와 2차(Gemini)를 나란히 띄운다.

```
=== E006-005-001-001 ===

[1차 Vision] 412 자
(모양만 보고 뽑은 글자들)

[2차 Gemini] 388 자
(문맥에 맞게 고친 결과)

'[?]' 표시: 4 곳
'□' 표시: 1 곳
```

**세 가지를 본다.**

첫째, **무엇이 나아졌는가.** 행 순서가 바로잡혔는지, 닮은 글자 오독이 고쳐졌는지, 표의 칸 구조가 살아났는지 본다.

둘째, **무엇이 생겨났는가.** 1차에 없던 글자가 2차에 있다면 그것이 어디서 왔는지 물어야 한다. 이미지에서 읽어 낸 것이면 교정이고, **아니면 환각이다.** 둘의 차이는 원본 이미지를 봐야만 갈린다. 화면 옆에 사료 사진을 띄워 두고 본다.

셋째, **`[?]`와 `□`가 찍혔는가.** ⑤의 7번 지침이 시킨 표시다. 하나도 없이 매끄러운 결과가 나왔다면 오히려 의심스럽다. 흐린 필사면에서 모든 글자를 확신한다는 것은 자연스럽지 않다.

글자 수가 1차보다 줄어드는 것은 흔한 일이다. 인장 테두리에서 잘못 잡힌 영문자 같은 것이 정리되기 때문이다.

여기까지는 숫자만 본 것이다. **⑧-보충에서 원본 사진을 함께 띄우고 직접 찾는다.**


In [ ]:
import os
import json

result_files = []
for name in sorted(os.listdir(OUTPUT_DIR)):
    if name.endswith('.json') and not name.endswith('_full.json'):
        result_files.append(name)

# 첫 장을 본다 (다른 장을 보려면 [0] 을 [1] 이나 [2] 로 바꾼다)
target = result_files[0]
data = json.load(open(os.path.join(OUTPUT_DIR, target), encoding='utf-8'))

vision = data['vision_raw']
gemini = data['gemini_corrected']

print('===', target.replace('.json', ''), '===')
print()
print('[1차 Vision]', len(vision), '자')
print(vision[:600])
print()
print('[2차 Gemini]', len(gemini), '자')
print(gemini[:600])
print()
print("'[?]' 표시:", gemini.count('[?]'), '곳')
print("'□' 표시:", gemini.count('□'), '곳')

### ⑧-보충 ★ 직접 찾아본다 — 오늘의 핵심

**여기부터가 3단계, 곧 연구자 검토다.** 앞의 두 단계는 기계가 했고, 이 단계는 사람이 한다.

아래 셀은 **원본 사진과 교정문을 한 화면에 띄운다.** 사진을 위에 두고 글을 아래에 두었으니, 눈을 위아래로 옮기며 맞춰 보면 된다.

**세 가지를 찾는다.**

1. **나아진 곳** — 행 순서가 바로잡혔는지, 닮은 글자 오독이 고쳐졌는지, 표의 칸이 살아났는지.
2. **생겨난 곳** — 1차에 없던 글자가 2차에 있다면 어디서 왔는지 묻는다. 사진에서 읽어 낸 것이면 교정이고, **사진에 없으면 환각이다.**
3. **`[?]`와 `□`** — ⑤의 지침이 시킨 표시다. 하나도 없이 매끄러우면 오히려 의심한다.

`번호`를 `0`, `1`, `2`로 바꾸면 다른 장을 볼 수 있다. **세 장 모두 보는 것을 권한다.** 인쇄 활자면과 흐린 필사면은 결과가 크게 다르다.


In [ ]:
import os
import json
from IPython.display import display, Image

# ★ 볼 사료를 고른다 : 0, 1, 2
번호 = 0

파일들 = sorted(n for n in os.listdir(OUTPUT_DIR)
              if n.endswith('.json') and not n.endswith('_full.json'))

대상 = 파일들[번호]
자료 = json.load(open(os.path.join(OUTPUT_DIR, 대상), encoding='utf-8'))

print('===', 대상.replace('.json', ''), '===')
print()

# (1) 원본 사진
사진 = os.path.join(INPUT_DIR, 자료['file_path'])
if os.path.exists(사진):
    display(Image(filename=사진, width=620))
else:
    print('원본 사진을 찾지 못했다. ③의 INPUT_DIR을 확인한다.')

# (2) 교정문 전문
print()
print('--- 2차 Gemini 교정문 (전문) ---')
print()
print(자료['gemini_corrected'])

#### ⑧-보충 ② 한 낱말을 두 단계에서 찾아본다

눈으로 훑기 어려우면 낱말 하나를 정해 두 단계에서 각각 찾아본다. **1차에는 없는데 2차에는 있다면, 그 자리가 검토해야 할 곳이다.**

첫 장에서 `賞勳局`을 넣어 보기 바란다. 원본에는 상훈국이 적혀 있으나 모델이 다른 기관 이름으로 바꾸어 놓은 자리가 있다. 이런 오류는 문장이 자연스러워서 **원본을 보지 않으면 발견되지 않는다.**

기관명이 바뀌면 그 문서가 무엇을 증명하는 자료인지가 통째로 달라진다. 검토를 생략할 수 없는 이유가 여기에 있다.


In [ ]:
# ★ 찾을 낱말을 바꿔 가며 확인한다
찾을말 = '賞勳局'

일차 = 자료['vision_raw']
이차 = 자료['gemini_corrected']

print('찾을 낱말 :', 찾을말)
print('1차 Vision :', 일차.count(찾을말), '번')
print('2차 Gemini :', 이차.count(찾을말), '번')
print()

def 앞뒤보기(글, 낱말, 이름):
    자리 = 글.find(낱말)
    if 자리 < 0:
        print('[' + 이름 + '] 없음')
        return
    처음 = max(0, 자리 - 30)
    끝 = min(len(글), 자리 + len(낱말) + 30)
    print('[' + 이름 + '] ...' + 글[처음:끝].replace('\n', ' ') + '...')

앞뒤보기(일차, 찾을말, '1차')
앞뒤보기(이차, 찾을말, '2차')

#### ⑧-보충 ③ 찾은 것을 적어 남긴다

머릿속으로만 확인하고 넘어가면 남는 것이 없다. **찾은 자리를 적어 파일로 남긴다.** 아래 셀의 따옴표 세 개 사이에 자유롭게 쓰면 된다.

이 메모가 곧 판독의 근거가 된다. 나중에 이 텍스트를 인용할 때, 어디까지가 판독이고 어디부터가 추정인지 밝히는 자료가 이것이다.


In [ ]:
메모 = """
[검토 메모]

1. 나아진 곳
   -

2. 의심스러운 곳 (원본에 없는데 생겨난 글자)
   -

3. [?] · □ 가 찍힌 자리
   -

4. 판단
   -
"""

import os
파일이름 = 대상.replace('.json', '_검토메모.txt')
경로 = os.path.join(OUTPUT_DIR, 파일이름)

with open(경로, 'w', encoding='utf-8') as f:
    f.write('사료: ' + 대상.replace('.json', '') + '\n')
    f.write(메모)

print('저장했다 →', 경로)
print()
print(메모)

### ⑨ [실행만] 비용 감각 익히기

**이번 실습은 무료 등급이라 청구액이 없다.** 그래도 토큰이 얼마나 쓰였는지는 그대로 나오므로, **유료 등급이었다면 얼마였을지**를 환산해 본다. 본인 사료로 넘어갈 때를 위한 감각이다.

```
호출: 3 회
입력 토큰: 4,820
출력 토큰: 1,340

이번 실습 청구액: ₩0 (무료 등급)
유료 등급이었다면: 약 ₩7
사료 900장 환산   : 약 ₩2,100
```

**토큰**은 요금을 매기는 단위로, 대략 글자 한두 개에 해당한다. 이미지도 토큰으로 환산된다. 입력보다 출력이 훨씬 비싸다.

사료 한 장에 몇 원 수준이므로, 미간행 자료를 유료 등급으로 옮겨 수백 장을 처리하더라도 부담이 큰 규모는 아니다. **무료 등급을 벗어나는 이유는 요금이 아니라 데이터 정책이다.**

여기 나오는 값은 코드에 적어 둔 단가로 계산한 **어림값**이다. 단가는 바뀔 수 있으므로, 본인 사료를 대량으로 처리하기 전에는 [공식 요금 페이지](https://ai.google.dev/gemini-api/docs/pricing)를 확인한다.

In [ ]:
# gemini-2.5-flash 단가 (100만 토큰당 달러)
PRICE_IN = 0.30
PRICE_OUT = 2.50
USD_TO_KRW = 1400

cost_usd = total_in * PRICE_IN / 1000000 + total_out * PRICE_OUT / 1000000
cost_krw = cost_usd * USD_TO_KRW

print('호출:', done, '회')
print('입력 토큰:', format(total_in, ','))
print('출력 토큰:', format(total_out, ','))
print()

# 이번 실습은 무료 등급이므로 청구되지 않는다
print('이번 실습 청구액: ₩0 (무료 등급)')
print('유료 등급이었다면: 약 ₩' + str(round(cost_krw)))

# 본인 사료 규모로 환산해 본다
if done > 0:
    per_page = cost_krw / done
    print('사료 900장 환산   : 약 ₩' + format(round(per_page * 900), ','))

### ⑩ 다음 단계

이제 사료 한 장마다 JSON 하나에 1차와 2차가 모두 담겼다. `_gemini_final.txt`는 사람이 읽으라고 따로 뽑아 둔 것이다.

**그런데 이것은 아직 판독문이 아니다.**

모델이 채운 자리 가운데 원본에 없는 글자가 섞여 있을 수 있고, 그것은 **원본 이미지와 한 글자씩 대조해야만 드러난다.** 이 대조를 거치지 않은 텍스트는 학술적으로 쓸 수 없다. 인용하려면 어디까지가 판독이고 어디부터가 추정인지 밝힐 수 있어야 하기 때문이다.

읽히지 않는 글자는 사람이 `□`로 남긴다. 모델이 채운 자리는 사람이 되짚는다. **파이프라인의 주체는 연구자다.**

---

**각자의 사료로 옮길 때**

이 노트북은 기본형이다. 자료가 다르면 고쳐 쓴다.

바꿔도 되는 것이 셋이다. step1 ③의 **언어 힌트**를 자기 자료의 언어에 맞춘다. 여기 ⑤의 **교정 지침**에 자기 자료의 시대와 서식, 자주 나오는 인명과 지명을 적는다. 1단계 **엔진**도 자료에 따라 다른 도구로 바꿀 수 있다.

바꾸지 않는 것도 셋이다. **추출·교정·연구자 검토의 3단계 구조**, **원본 대조**, 그리고 못 읽은 자리를 채우지 않고 **□로 남기는 표시**다.

파이프라인은 도구의 이름이 아니라 순서의 이름이다. 도구는 바뀌어도 순서는 남는다.

---

**이어서 할 수 있는 것**

* `_gemini_final.txt`를 원본과 대조하며 직접 손보기
* 본문에서 인명·지명을 뽑아 세어 보기
* 여러 사료의 텍스트를 모아 검색해 보기

step1의 `_full.json`에 남겨 둔 좌표를 쓰면, 판독문의 한 글자를 눌렀을 때 원본 사진의 그 자리가 표시되는 디지털 판본으로 나아갈 수 있다. 거기서부터는 OCR이 아니라 편찬의 영역이다.